In [31]:
import yfinance  as yf
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd



In [32]:
data = yf.download("HDFCBANK.NS", start= "2018-01-01", end= "2025-01-01")

[*********************100%***********************]  1 of 1 completed


In [33]:
data.columns = data.columns.get_level_values(0)

In [42]:
# ==============================
# FEATURE ENGINEERING
# ==============================

# Moving averages
data['MA20'] = data['Close'].rolling(20).mean()
data['MA50'] = data['Close'].rolling(50).mean()

# Trend
data['Trend'] = (data['MA20'] > data['MA50']).astype(int)
data['MA_diff'] = data['MA20'] - data['MA50']

# Momentum
data['Return'] = data['Close'].pct_change()
data['Momentum_3'] = data['Close'].pct_change(3)
data['Momentum_5'] = data['Close'].pct_change(5)

# Volatility
data['Volatility'] = data['Close'].rolling(20).std()
data['Range'] = data['High'] - data['Low']

# Volume (if available)
data['Volume_MA20'] = data['Volume'].rolling(20).mean()
data['Volume_Spike'] = data['Volume'] / data['Volume_MA20']

# Price Action
data['Candle_Body'] = data['Close'] - data['Open']
data['Body_Ratio'] = data['Candle_Body'] / data['Range']

# Breakout
data['High_20'] = data['High'].rolling(20).max()
data['Breakout'] = (data['Close'] > data['High_20']).astype(int)

# ==============================
# BETTER TARGET
# ==============================

data['Target'] = (data['Return'].shift(-1) > 0.005).astype(int)

In [43]:
data = data.dropna()

In [44]:
data.head()

Price,Close,High,Low,Open,Volume,MA20,MA50,Trend,MA_diff,Return,...,Momentum_5,Volatility,Range,Volume_MA20,Volume_Spike,Candle_Body,Body_Ratio,High_20,Breakout,Target
Date,,,,,,,,,,,,,,,,,,,,,
2018-05-25,468.666656,470.369892,464.280288,464.303614,4759588,464.173598,452.401170,1,11.772428,0.011430,...,-0.001715,6.756094,6.089605,5775041.2,0.824165,4.363043,0.716474,481.802549,0,1
2018-05-28,476.482849,478.769381,468.036734,468.456719,5103940,465.559505,453.230383,1,12.329121,0.016678,...,0.024661,6.253085,10.732647,5741387.8,0.888973,8.026131,0.747824,481.802549,0,0
2018-05-29,474.243073,478.932768,473.356479,475.502998,3351244,466.589607,453.938737,1,12.650870,-0.004701,...,0.021458,5.871664,5.576288,5678472.4,0.590166,-1.259925,-0.225943,481.802549,0,1
2018-05-30,477.917816,479.002724,470.439949,471.233209,4866996,467.509467,454.850313,1,12.659153,0.007749,...,0.041093,6.140836,8.562775,5566819.8,0.874287,6.684607,0.780659,481.802549,0,1
2018-05-31,502.361359,507.186688,488.401996,493.098169,39639192,469.670164,456.277591,1,13.392574,0.051146,...,0.084146,9.645957,18.784692,7341140.8,5.399596,9.263190,0.493124,507.186688,0,0


In [45]:
# =========================================
features = [
    'Trend', 'MA_diff',
    'Momentum_3', 'Momentum_5',
    'Volatility', 'Range',
    'Volume_Spike',
    'Body_Ratio',
    'Breakout'
]

X = data[features]
y = data['Target']

# =========================================
# STEP 7: TRAIN-TEST SPLIT (TIME SERIES)
# =========================================
split = int(len(data) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]



In [46]:
# =========================================
# STEP 8: MODEL
# =========================================
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)

# Train
model.fit(X_train, y_train)



RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=300,
                       random_state=42)

In [47]:
# Predict
y_pred = model.predict(X_test)

# =========================================
# STEP 9: EVALUATION
# =========================================
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


Accuracy: 0.598159509202454


In [48]:

# =========================================
# STEP 10: FEATURE IMPORTANCE
# =========================================
importance = pd.Series(model.feature_importances_, index=X.columns)
print("\nFeature Importance:\n")
print(importance.sort_values(ascending=False))


Feature Importance:

Price
Body_Ratio      0.151007
Momentum_3      0.145226
MA_diff         0.145126
Momentum_5      0.142868
Range           0.137753
Volume_Spike    0.135512
Volatility      0.133017
Trend           0.009491
Breakout        0.000000
dtype: float64
